In [0]:
%pip install google-cloud-storage

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../tests/test_silver_layer

In [0]:
%run ../utils/gcp_setup

In [0]:
import os
from google.cloud import storage
import pyspark.sql.functions as F
import logging
import sys

# Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("HolidaysBronzeIngestion")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("credentials_path", "")
dbutils.widgets.text("gcs_bucket_name", "")
dbutils.widgets.text("gcs_holidays_file_path", "")
dbutils.widgets.text("volume_output_path", "")
dbutils.widgets.text("bronze_holidays_table", "")

credentials_path = dbutils.widgets.get("credentials_path") or "/Workspace/Shared/service-account.json"
gcs_bucket_name = dbutils.widgets.get("gcs_bucket_name") or "prefect-bucket-latypov"
gcs_holidays_file_path = dbutils.widgets.get("gcs_holidays_file_path") or "chicago_holidays_data/chicago_holidays_2025_2026.parquet"
dbfs_chicago_holidays_path = dbutils.widgets.get("dbfs_chicago_holidays_path") or f"/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026"
bronze_holidays_table = dbutils.widgets.get("bronze_holidays_table") or "chicago_taxi_data.bronze.bronze_holidays"

**Google Cloud credentials Set Up**

In [0]:

# Set up Google Cloud service account keys 

logger.info(f"Set up Google Cloud service account keys")
setup_gcp_creds(credentials_path=credentials_path)


2026-05-13 10:24:34 - INFO - Set up Google Cloud service account keys
2026-05-13 10:24:34 - INFO - Found service account file
2026-05-13 10:24:35 - INFO - Successfully authenticated. Found 2 buckets.


**Chicago Holidays Data Ingestion to Databricks Unity Catalog Volume**

In [0]:

try:
    # 1. Setup GCS client
    client = storage.Client()

    bucket = client.get_bucket(gcs_bucket_name)
    logger.info(f"GCS bucket: {bucket}")
    blob = bucket.blob(gcs_holidays_file_path)
    logger.info(f"Blob in GCS bucket: {blob}")

    # 2. Path to Unity Catalog Volume
    os.makedirs(dbfs_chicago_holidays_path, exist_ok=True)
    output_filename = gcs_holidays_file_path.split('/')[-1]
    logger.info(f"Output filename: {output_filename}")
    destination_path = f"{dbfs_chicago_holidays_path}/{output_filename}"
    logger.info(f"Destination Path: {destination_path}")

    # 3. Write to Unity Catalog Volume

    with open(destination_path, "wb") as f:
        blob.download_to_file(f)

    logger.info(f"File with holiday data ingestested to {destination_path}")
except Exception as e:
    logger.error(f"File ingestestion failed: {e}")
    dbutils.notebook.exit(f"File ingestestion failed: {e}")

2026-05-13 10:24:37 - INFO - GCS bucket: <Bucket: prefect-bucket-latypov>
2026-05-13 10:24:37 - INFO - Blob in GCS bucket: <Blob: prefect-bucket-latypov, chicago_holidays_data/chicago_holidays_2025_2026.parquet, None>
2026-05-13 10:24:37 - INFO - Output filename: chicago_holidays_2025_2026.parquet
2026-05-13 10:24:37 - INFO - Destination Path: /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-05-13 10:24:37 - INFO - File with holiday data ingestested to /Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet


**Data Quality Check**

Making automated bronze layer quality checks on the raw incoming GCS data



In [0]:
df_raw_holidays = spark.read.parquet(destination_path)

required_columns = ["date", "holiday_name"]
try: 
    logger.info("Starting Testing Holidays Bronze Ingestion...")

    #validate_duplicates(df_raw_holidays, "date")

    validate_schema(df_raw_holidays, required_columns)

    #validate_no_nulls(df_raw_holidays, "date")

    #validate_data_type(df_raw_holidays, "date", "timestamp_ntz")

    validate_date_range(df_raw_holidays, "date", min_date="2025-01-01", max_date="2027-01-01")
    logger.info("DQ passed")
except Exception as e:
    error_message = str(e)
    logger.error(f"Aborting Job: {error_message}")
    
    dbutils.notebook.exit(f"{error_message}\nDQ checks FAILED")



2026-05-13 10:24:40 - INFO - Starting Testing Holidays Bronze Ingestion...
2026-05-13 10:24:41 - INFO - validate_schema - PASSED (All required columns present: ['date', 'holiday_name'])
2026-05-13 10:24:41 - INFO - validate_date_range - PASSED (All dates in column date are within 2025-01-01-2027-01-01)
2026-05-13 10:24:41 - INFO - DQ passed


**Save Chicago Holidays Data File as Delta Table**

In [0]:
from delta.tables import DeltaTable


df_bronze_holidays = df_raw_holidays.withColumn("ingestion_timestamp", F.current_timestamp()).\
                                    withColumn("source_file", F.col("_metadata.file_path"))
# 1. Check if the table exists
if spark.catalog.tableExists(bronze_holidays_table):
    logger.info("Table exists. Performing MERGE (Upsert)...")
    
    target_table = DeltaTable.forName(spark, bronze_holidays_table)
    
    # 2. Perform the Merge
    (target_table.alias("target")
     .merge(
         df_bronze_holidays.alias("source"),
         "target.date = source.date" # Match on holiday date
     )
     .whenMatchedUpdateAll()        # Update existing records
     .whenNotMatchedInsertAll()     # Insert new records
     .execute())
else:
    logger.info("Table does not exist. Creating and saving for the first time...")
    df_bronze_holidays.write.format("delta").saveAsTable(bronze_holidays_table)

2026-05-13 10:24:45 - INFO - Table exists. Performing MERGE (Upsert)...


In [0]:
# df_bronze_holidays = df_raw_holidays.withColumn("ingestion_timestamp", F.current_timestamp()).\
#                                     withColumn("source_file", F.col("_metadata.file_path"))
# try:
#     df_bronze_holidays.write.format("delta")\
#                             .mode("append")\
#                             .option("mergeSchema", "true")\
#                             .saveAsTable(bronze_holidays_table)
# except Exception as e:
#     logger.info(f"Write Error: {e}")
#     dbutils.notebook.exit(error_message)


# # Display Result
display(spark.sql(f"SELECT * FROM {bronze_holidays_table} ORDER BY ingestion_timestamp DESC"))

date,holiday_name,ingestion_timestamp,source_file
2026-07-04T00:00:00.000,Independence Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-11-26T00:00:00.000,Thanksgiving Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-02-16T00:00:00.000,Washington's Birthday,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-07-03T00:00:00.000,Independence Day (observed),2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-11-03T00:00:00.000,Election Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-11-11T00:00:00.000,Veterans Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2025-10-13T00:00:00.000,Columbus Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-02-12T00:00:00.000,Lincoln's Birthday,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-03-02T00:00:00.000,Casimir Pulaski Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
2026-06-19T00:00:00.000,Juneteenth National Independence Day,2026-05-13T10:24:45.883Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_holidays_2025_2026/chicago_holidays_2025_2026.parquet
